# 01 · The turbine, its power curve and its sensors

**Purpose.** Understand what produces the power we forecast: a 0.6 m helical Savonius rotor with flow cones, a 4:1
gear to a small PMG charging a 12.8 V LiFePO4 battery, and an 18 W two-axis tracking PV panel.

```
 wind ─► rotor Cp(λ) ─► gear ─► PMG ─► rectifier ─► charge controller ─► battery ─► 12 V loads (Pi 5, ESP32 …)
  sun ─► PV + tracker ─► MPPT ──────────────────────────────┘   (full battery ⇒ PV curtailed)
```
**Key takeaways**
1. One Cp(λ) curve reproduces the whole design table.
2. The generator charges only above ≈ 4.5 m/s, and the 200 rpm brake trips near 8.2 m/s: wind power is a narrow, steep band.
3. A full battery curtails PV: delivered power depends on the battery, not only on the sun.

In [1]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aerosolar.config import Config, REPO_ROOT
from aerosolar.sim.realism import _style, S1, S2, S3, INK, INK2
_style(); plt.rcParams['figure.dpi'] = 110
cfg = Config.load()
DATA = REPO_ROOT / 'data'
pd.set_option('display.width', 140); pd.set_option('display.max_columns', 30)

## 1. Rotor aerodynamics

With x = λ/λ_opt: $C_p(x) = C_{p,\max}\,x\,\frac{(k+1)-x^k}{k}$, peaking at the design point (λ = 0.785, Cp = 0.20).
Torque $T = \tfrac12\rho A R v^2 C_q$ [N·m], power $P = \tfrac12\rho A v^3 C_p$ [W].

> **Assumption.** The roll-off exponent k = 2 is typical of helical Savonius rotors but unmeasured (CALIBRATION.md step 5).

In [2]:
from aerosolar.sim.aero import AeroParams, cp_of_tsr
p = AeroParams.from_config(cfg.turbine)
tsr = np.linspace(0, 1.5, 300)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(tsr, cp_of_tsr(tsr, p), color=S1, label='Cp'); ax.plot(tsr, cp_of_tsr(tsr, p) * p.cone_gain, color=S2, label='with cones')
ax.scatter([0.785], [0.2], color=INK, zorder=3, label='design point'); ax.set_xlabel('tip-speed ratio λ'); ax.set_ylabel('Cp'); ax.legend(); plt.show()

**What to notice.** The design point is the peak; torque is highest at standstill (easy start) and vanishes at runaway (λ ≈ 1.36).

## 2. The steady-state power curve

In [3]:
from aerosolar.sim.realism import model_power_curve
v = np.linspace(0, 11, 45)
pc = model_power_curve(cfg, v)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(v, pc, color=S1); ax.axvspan(8.2, 11, color=S2, alpha=0.12, label='overspeed brake trips (> 200 rpm)')
ax.set_xlabel('hub wind [m/s]'); ax.set_ylabel('electrical power to battery [W]'); ax.legend(fontsize=8); plt.show()

**What to notice.** Nothing below ≈ 4.5 m/s (the PMG voltage must exceed the battery), a cubic rise, and in real
operation the brake above ≈ 8.2 m/s. A 1 m/s wind-forecast error near 6 m/s changes power by ~50 %: this is why wind is
the hard part of the forecast.

> **Caution (design finding).** Both limits follow from the design numbers (K, battery voltage, trip setting); see ASSUMPTIONS.md.

## 3. What the Pi measures
The forecast is trained on the INA219 readings (wind and PV power), the anemometer, the BH1750 and the battery state — exactly the 19-field ESP32 packet:

In [4]:
from aerosolar.sim.engine import Engine
r = Engine(cfg, {'start': '2026-04-12T06:00:00'}, seed=3).step(4 * 3600)
r.packets.drop_duplicates('seq').head(3).T

,0,1,2
ts,1775959200350,1775959201350,1775959202350
seq,0,1,2
wind_ms,0.667,4.669,7.337
rpm,120.0,180.0,180.0
hall_count,2.0,3.0,3.0
bus_v,13.268,13.26,13.272
current_a,0.199,0.5394,0.6284
power_w,2.64,7.152,8.34
pv_power_w,0.0,0.0,0.0
batt_v,13.24,13.256,13.26


**What to notice.** Quirks the cleaning step handles: 0.667 m/s anemometer steps, ±60 rpm Hall steps, BH1750 saturation at
65 535 lx (≈ 546 W/m²), DS18B20 glitch codes, Wi-Fi losses and an ESP32 clock offset.

## What we learned
* The power curve is narrow and steep for wind, smooth and predictable for PV.
* Delivered PV depends on the battery (curtailment), which the forecast must model.

## What changes with real data
Measure the Cp curve, K and the loads (CALIBRATION.md); the forecast is then retrained on the measured stream.

## References
* Romano, Patterson, Candès (2019). *Conformalized quantile regression.* NeurIPS.
* Ke et al. (2017). *LightGBM: a highly efficient gradient boosting decision tree.* NeurIPS.
* Yang et al. (2020). *Verification of deterministic solar forecasts.* Solar Energy 210 (clear-sky index, smart persistence, skill scores).
* Hong et al. (2016). *Probabilistic energy forecasting: GEFCom2014 and beyond.* IJF 32.
* Haurwitz (1945). *Insolation in relation to cloudiness and cloud density.* J. Meteorology.
* Kamoji, Kedare, Prabhu (2009). *Performance tests on helical Savonius rotors.* Renewable Energy 34.